In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies','Glucose','BloodPressure','SkinThickness','Insulin','BMI','DiabetesPedigreeFunction','Age','Outcome']
df = pd.read_csv(url, names=columns)
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [13]:
from sklearn.model_selection import train_test_split
X=df.drop('Outcome',axis=1)
y=df['Outcome']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
X_train = X_train.copy()
X_test = X_test.copy()

This dataset is in need of some preprocessing.
Values that are 0 in columns other than Pregnancies and Outcome are actually NaN

In [14]:
nancols = ['Glucose','BloodPressure','SkinThickness','Insulin','BMI','DiabetesPedigreeFunction','Age']
X_train[nancols] = X_train[nancols].replace(0,np.nan)
X_train.head()
X_test[nancols] = X_test[nancols].replace(0,np.nan)
X_test.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
668,6.0,98.0,58.0,33.0,190.0,34.0,0.430,43.0
324,2.0,112.0,75.0,32.0,125.0,35.7,0.148,21.0
624,2.0,108.0,64.0,29.0,125.0,30.8,0.158,21.0
690,8.0,107.0,80.0,29.0,125.0,24.6,0.856,34.0
473,7.0,136.0,90.0,29.0,125.0,29.9,0.210,50.0


In [20]:
#time to impute the NaNs with the median
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(missing_values=np.nan, strategy="median")
X_train_imp = imputer.fit_transform(X_train)
X_test_imp = imputer.transform(X_test)

In [41]:
#scaling values
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_std_lr = pd.DataFrame(scaler.fit_transform(X_train_imp), columns=X_train.columns, index=X_train.index)
X_test_std_lr = pd.DataFrame(scaler.transform(X_test_imp), columns=X_test.columns, index=X_test.index)
X_train_std_knn = X_train_std_lr.copy()
X_test_std_knn = X_test_std_lr.copy()
X_train_std.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
60,-0.526397,-1.256881,-0.018995,0.034298,-0.175620,-0.007450,-0.490735,-1.035940
618,1.588046,-0.326051,0.808174,-0.560583,-0.175620,-0.599092,2.415030,1.487101
346,-0.828460,0.571536,-2.169636,-1.155463,-0.652193,-0.526941,0.549161,-0.948939
294,-1.130523,1.302903,-1.838768,0.034298,-0.175620,-1.508200,-0.639291,2.792122
231,0.681856,0.405316,0.642740,0.986106,2.604392,1.998360,-0.686829,1.139095


MODEL 1: LOGISTIC REGRESSION FROM SCRATCH

In [42]:
class LogisticRegression:
  def __init__(self, lr=0.01, iterations=5000, threshold=0.5):
    self.lr = lr
    self.iterations = iterations
    self.threshold = threshold

  @staticmethod
  def sigmoid(z):
    z = np.clip(z,-500,500)
    return (1.0/(1.0 + np.exp(-z)))

  def fit(self, X, y):
    self.w = np.zeros(X.shape[1])
    self.b=0
    for i in range(self.iterations):
      z = X@self.w + self.b
      p = self.sigmoid(z)
      error = p - y
      grad_w = (X.T@error)/X.shape[0]
      grad_b = error.mean()
      self.w = self.w - self.lr*grad_w
      self.b = self.b - self.lr*grad_b
      cost_function = -np.mean(y*np.log(p) + (1-y)*np.log(1-p))
      if(i%10==0):
        print(f"Iteration {i} : Cost Function : {cost_function}")
    return self
  def predict(self,X):
    return (self.sigmoid(X@self.w  +self.b) >= self.threshold).astype(int)

In [43]:
y_pred = LogisticRegression(0.05,10000).fit(X_train_std,y_train).predict(X_test_std)

Iteration 0 : Cost Function : 0.6931471805599453
Iteration 10 : Cost Function : 0.6299138159637914
Iteration 20 : Cost Function : 0.5898629266053922
Iteration 30 : Cost Function : 0.5631104035359222
Iteration 40 : Cost Function : 0.5442968089317138
Iteration 50 : Cost Function : 0.5304735772487611
Iteration 60 : Cost Function : 0.5199472710687983
Iteration 70 : Cost Function : 0.5116969414206818
Iteration 80 : Cost Function : 0.5050779601763292
Iteration 90 : Cost Function : 0.4996660223766785
Iteration 100 : Cost Function : 0.49517142495949884
Iteration 110 : Cost Function : 0.49138994185213253
Iteration 120 : Cost Function : 0.4881735439026292
Iteration 130 : Cost Function : 0.485412308917103
Iteration 140 : Cost Function : 0.4830228754131396
Iteration 150 : Cost Function : 0.4809408529413774
Iteration 160 : Cost Function : 0.47911569878495763
Iteration 170 : Cost Function : 0.47750717550190014
Iteration 180 : Cost Function : 0.47608284780399285
Iteration 190 : Cost Function : 0.4748

In [44]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
def metrics_row(name, y_true, y_pred):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred),
    }
print(metrics_row("Logistic Regression (scratch)", y_test, y_pred))

{'Model': 'Logistic Regression (scratch)', 'Accuracy': 0.7532467532467533, 'Precision': 0.6666666666666666, 'Recall': 0.6181818181818182, 'F1': 0.6415094339622641}


MODEL 2: K-NN scratch

In [47]:
class KNearestNeighbours:
  def __init__(self,k=5):
    self.k = k
  def fit(self,X,y):
    self.X_train = X
    self.y_train = y.astype(int) # Convert y to integer type
    return self
  def predict(self,X):
    preds = np.empty(X.shape[0],dtype=int)
    for i,x in enumerate(X):
      dists = np.sqrt(np.sum((self.X_train-x)**2,axis=1))
      nearest_dists_index = np.argsort(dists)[:self.k]
      nearest_labels = self.y_train[nearest_dists_index]
      bruh = np.bincount(nearest_labels)
      preds[i] = np.argmax(bruh)
    return preds

In [52]:
y_pred = KNearestNeighbours(4).fit(X_train_std_knn.values, y_train.values).predict(X_test_std_knn.values)

In [53]:
print(metrics_row("KNN (scratch)" , y_test, y_pred))

{'Model': 'KNN (scratch)', 'Accuracy': 0.7272727272727273, 'Precision': 0.6511627906976745, 'Recall': 0.509090909090909, 'F1': 0.5714285714285714}
